### Евсеев ИУ5-64Б Семинар 6

In [1]:
# -*- coding: utf-8 -*-
import threading, random, time

# Обедающие философы
# 5 мудрецов и 5 палочек для лапши(или вилочек). Для еды нужно ухватить две палочки.
#
# Для устранения блокировки (Deadlock) используется механизм мьютексов (lock).
# Procedure is to do block while waiting to get first fork, and a nonblocking
# acquire of second fork.  If failed to get second fork, release first fork,
# swap which fork is first and which is second and retry until getting both.

class Filosoff(threading.Thread):

    running = True

    def __init__(self, xname, forkOne, forkTwo):
        #threading.Thread.__init__(self)
        super().__init__()
        self.name = xname
        self.forkOnLeft = forkOne
        self.forkOnRight = forkTwo
        self.f_eat_count =0 

    def run(self):
        while self.running:
            #  Мудрец думает (на самом деле спит)
            time.sleep(random.uniform(2,5))
            print (self.name, 'голодный.')
            self.dine()
        else:
            print (self.name,' поел ', self.f_eat_count," раз" )
            return

    def dine(self):
        fork1, fork2 = self.forkOnLeft, self.forkOnRight

        while self.running:
            fork1.acquire(True)
			# простой вариант контроллера_!_жадный_
            #fork2.acquire(True)
            #break
            # анти-дедлок -vvvv- (deadlock prevent section)
            locked = fork2.acquire(False)
            if locked: break
            fork1.release()
            # анти-дедлок -^^^^-
            # улучшим эффективность захвата -vvvv-
            print (self.name, 'поменял порядок захвата.')
            fork1, fork2 = fork2, fork1
            # попробуем начать захват с другого ресурса -^^^^-
        else:
            return
        self.f_eat_count += 1
        print (self.name, 'начал есть.')
        time.sleep(random.uniform(3,6))
        print (self.name, 'наелся и думает.')
        fork1.release()
        fork2.release()

In [3]:
def DiningFilosofs(numfilos):
    if (numfilos < 2) or (numfilos > 12) :
       print('Число философов от 2 до 12!')
       return

    # создание семафоров ресурсов (forks) в виде мьютексов
    forks = [threading.Lock() for _ in range(numfilos)]

    philosopherNames = ('1:Кант:','2:Маркс:','3:Платон:','4:Руссо:','5:Сократ:',
                        '6:Пифагор:','7:Гегель:','8:Вольтер:','9:Декарт:',
                        '10:ЛаоЦзы:','11:Рассел:','12:Лев:')
    
    # список имен по-английски (вдруг пригодится?)
    # philosopherNames = ('1:Kant:','2:Marx:','3:Plato:','4:Russo:','5:Sockrat:',
    #                    '6:Pifagor:','7:Gheghel:','8:Voltair:','9:Dekart:',
    #                    '10:Lao:','11:Russel:')

    philosophers = [Filosoff(philosopherNames[i], forks[i%numfilos],
                                forks[(i+1)%numfilos]) for i in range(numfilos)]

    random.seed(1537)
    modelstart = time.perf_counter_ns()
    Filosoff.running = True
    for p in philosophers: p.start()
    time.sleep(40)
    Filosoff.running = False
    # Ожидание завершения потоков
    for p in philosophers: p.join()
    modelfini = time.perf_counter_ns()
    print ("== Все закончили! Время =", (modelfini-modelstart)/1000000, "msec")

In [4]:
DiningFilosofs(10)

3:Платон: голодный.
3:Платон: начал есть.
1:Кант: голодный.
1:Кант: начал есть.
4:Руссо: голодный.
2:Маркс: голодный.
5:Сократ: голодный.
5:Сократ: начал есть.
8:Вольтер: голодный.
8:Вольтер: начал есть.
10:ЛаоЦзы: голодный.
10:ЛаоЦзы: поменял порядок захвата.
7:Гегель: голодный.
7:Гегель: поменял порядок захвата.
9:Декарт: голодный.
6:Пифагор: голодный.
1:Кант: наелся и думает.
2:Маркс: поменял порядок захвата.
10:ЛаоЦзы: начал есть.
5:Сократ: наелся и думает.
6:Пифагор: начал есть.
3:Платон: наелся и думает.
4:Руссо: начал есть.
2:Маркс: начал есть.
8:Вольтер: наелся и думает.
7:Гегель: поменял порядок захвата.
9:Декарт: поменял порядок захвата.
1:Кант: голодный.
10:ЛаоЦзы: наелся и думает.
1:Кант: поменял порядок захвата.
9:Декарт: начал есть.
5:Сократ: голодный.
8:Вольтер: голодный.
8:Вольтер: поменял порядок захвата.
3:Платон: голодный.
2:Маркс: наелся и думает.
3:Платон: поменял порядок захвата.
1:Кант: начал есть.
6:Пифагор: наелся и думает.
7:Гегель: начал есть.
4:Руссо: наелся

Вариант с ограничением еды

In [5]:
# -*- coding: utf-8 -*-
import threading, random, time

# Обедающие философы
# 5 мудрецов и 5 палочек для суши(или вилочек). Для еды нужно ухватить две палочки.
# Для устранения блокировки (Deadlock) используется механизм мьютексов (lock).

class Filosofa(threading.Thread):
    #running = True

    def __init__(self, xname, forkOne, forkTwo):
        super().__init__()
        self.name = xname
        self.forkOnLeft = forkOne
        self.forkOnRight = forkTwo
        self.sushi_eaten =0 

    def run(self) -> None:
        global sushi
        while sushi > 0:
            #  Мудрец думает
            time.sleep(random.normalvariate(4,1))
            print (self.name, 'голодный.')
            # простой вариант !__#
            self.forkOnLeft.acquire(True)
            self.forkOnRight.acquire(True)
            # можно поесть
            if sushi > 0:
               sushi -= 1
               self.sushi_eaten += 1
               print (self.name, 'начал есть.')
            time.sleep(random.expovariate(1/3.0))
            print (self.name, 'наелся и думает.')
            self.forkOnLeft.release()
            self.forkOnRight.release()
        else:
            print (self.name,' поел ', self.sushi_eaten," раз" )
            return
        

In [6]:
def DiningFilosofas(numfilos):
    if (numfilos < 2) or (numfilos > 12) :
       print('Число философов от 2 до 12!')
       return
    
    # создание семафоров ресурсов (палочек) в виде мьютексов
    forks = [threading.Lock() for _ in range(numfilos)]

    philosophNames = ('1:Кант:','2:Маркс:','3:Платон:','4:Руссо:','5:Сократ:',
                      '6:Пифагор:','7:Гегель:','8:Вольтер:','9:Декарт:',
                      '10:ЛаоЦзы:','11:Рассел:','12:Лев:')

    philosofs = [Filosofa(philosophNames[i], forks[i%numfilos],
                           forks[(i+1)%numfilos]) for i in range(numfilos)]

    random.seed(1537)
    modelstart=time.perf_counter_ns()
    for p in philosofs: p.start()
    time.sleep(40)
    # Ожидание завершения потоков
    for p in philosofs: p.join()
    modelfini=time.perf_counter_ns()
    print ("== Все закончили! Время =", (modelfini-modelstart)/1000000, "msec")

In [7]:
sushi=50
DiningFilosofas(10)

2:Маркс: голодный.
2:Маркс: начал есть.
5:Сократ: голодный.
5:Сократ: начал есть.
8:Вольтер: голодный.
8:Вольтер: начал есть.
1:Кант: голодный.
6:Пифагор: голодный.
2:Маркс: наелся и думает.
1:Кант: начал есть.
10:ЛаоЦзы: голодный.
9:Декарт: голодный.
3:Платон: голодный.
3:Платон: начал есть.
1:Кант: наелся и думает.
10:ЛаоЦзы: начал есть.
10:ЛаоЦзы: наелся и думает.
5:Сократ: наелся и думает.
6:Пифагор: начал есть.
4:Руссо: голодный.
7:Гегель: голодный.
3:Платон: наелся и думает.
4:Руссо: начал есть.
4:Руссо: наелся и думает.
6:Пифагор: наелся и думает.
2:Маркс: голодный.
2:Маркс: начал есть.
1:Кант: голодный.
10:ЛаоЦзы: голодный.
2:Маркс: наелся и думает.
1:Кант: начал есть.
5:Сократ: голодный.
5:Сократ: начал есть.
4:Руссо: голодный.
3:Платон: голодный.
6:Пифагор: голодный.
2:Маркс: голодный.
8:Вольтер: наелся и думает.
7:Гегель: начал есть.
7:Гегель: наелся и думает.
5:Сократ: наелся и думает.
6:Пифагор: начал есть.
4:Руссо: начал есть.
8:Вольтер: голодный.
4:Руссо: наелся и думает

Вариант с официантом

In [8]:
# -*- coding: utf-8 -*-
import threading, random, time

# Обедающие философы
# 5 мудрецов и 5 палочек для суши(или вилочек). Для еды нужно ухватить две палочки.
# Для устранения блокировки (Deadlock) используется механизм мьютексов (lock).

class Oficiant:
    def __init__(self) -> None:
        self.mutex = threading.Lock()

    def ask_for_forks(self, left_fork, right_fork) -> None:
        with self.mutex:
            left_fork.acquire()
            print("+левую вилку взял")
            right_fork.acquire()
            print("+правую вилку взял\n")

    def release_forks(self, left_fork, right_fork) -> None:
        right_fork.release()
        print("-правую вилку положил")
        left_fork.release()
        print("-левую вилку положил\n")


class Filosofik(threading.Thread):
    #running = True

    def __init__(self, xname, forkOne, forkTwo, waiter: Oficiant):
        super().__init__()
        self.name = xname
        self.forkOnLeft = forkOne
        self.forkOnRight = forkTwo
        self.sushi_eaten =0
        self.waiter = waiter

    def run(self) -> None:
        global sushi
        while sushi > 0:
            #  Мудрец думает
            time.sleep(random.normalvariate(4,1))
            print (self.name, 'голодный.')
            print(f"{self.name} запросил официанта")
            self.waiter.ask_for_forks(self.forkOnLeft, self.forkOnRight)
            # можно поесть
            if sushi > 0:
               sushi -= 1
               self.sushi_eaten += 1
               print (self.name, 'начал есть.')
            time.sleep(random.expovariate(1/3.0))
            print(f"{self.name} отдал официанту")
            self.waiter.release_forks(self.forkOnLeft, self.forkOnRight)
            print (self.name, 'наелся и думает.')
        else:
            print (self.name,' поел ', self.sushi_eaten," раз" )
            return
        

In [9]:
def DiningFilosofik(numfilos):
    if (numfilos < 2) or (numfilos > 12) :
       print('Число философов от 2 до 12!')
       return
    
    # создание семафоров ресурсов (палочек) в виде мьютексов
    forks = [threading.Lock() for _ in range(numfilos)]

    philosophNames = ('1:Кант:','2:Маркс:','3:Платон:','4:Руссо:','5:Сократ:',
                        '6:Пифагор:','7:Гегель:','8:Вольтер:','9:Декарт:',
                        '10:ЛаоЦзы:','11:Рассел:','12:Лев:')
    waiter = Oficiant()
    philosofs = [Filosofik(philosophNames[i], forks[i%numfilos],
                           forks[(i+1)%numfilos], waiter) for i in range(numfilos)]
    random.seed(1537)
    modelstart=time.perf_counter_ns()
    for p in philosofs: p.start()
    # Ожидание завершения потоков
    for p in philosofs: p.join()
    modelfini=time.perf_counter_ns()
    print ("== Все закончили! Время =", (modelfini-modelstart)/1000000, "msec")

In [10]:
sushi=50
DiningFilosofik(10)

2:Маркс: голодный.
2:Маркс: запросил официанта
+левую вилку взял
+правую вилку взял

2:Маркс: начал есть.
5:Сократ: голодный.
5:Сократ: запросил официанта
+левую вилку взял
+правую вилку взял

5:Сократ: начал есть.
8:Вольтер: голодный.
8:Вольтер: запросил официанта
+левую вилку взял
+правую вилку взял

8:Вольтер: начал есть.
1:Кант: голодный.
1:Кант: запросил официанта
+левую вилку взял
6:Пифагор: голодный.
6:Пифагор: запросил официанта
2:Маркс: отдал официанту
-правую вилку положил
-левую вилку положил

2:Маркс: наелся и думает.
+правую вилку взял

1:Кант: начал есть.
10:ЛаоЦзы: голодный.
10:ЛаоЦзы: запросил официанта
9:Декарт: голодный.
9:Декарт: запросил официанта
3:Платон: голодный.
3:Платон: запросил официанта
1:Кант: отдал официанту
-правую вилку положил
-левую вилку положил

1:Кант: наелся и думает.
5:Сократ: отдал официанту
-правую вилку положил
-левую вилку положил

5:Сократ: наелся и думает.
+левую вилку взял
+правую вилку взял

6:Пифагор: начал есть.
+левую вилку взял
+праву

В модели однопоточной конкурентности избегаем состояний гонки, вызванных неатомарными операциями. В `asyncio` есть только один поток, который в каждый момент времени исполняет одну строку кода Python. Это означает, что, даже если операция неатомарна, она все равно будет доведена до конца и другие сопрограммы не смогут прочитать несогласованные данные.

Вместо нескольких потоков модифицировать переменную будут несколько задач.  Блокировки (Lock) помогут гарантировать, что модификации производятся в желаемом синхронизированном порядке.

Главное отличие от `lock` потока заключается в том, `Lock asyncio` – объекты, допускающие ожидание, которые приостанавливают выполнение сопрограммы, когда заблокированы. Это значит, что если сопрограмма ожидает освобождения блокировки, то может работать другой код. Кроме того, блокировки `asyncio` являются 
асинхронными контекстными менеджерами, и предпочтительно использовать их в сочетании с конструкцией `async with`.

Функция `asyncio.gather()` дожидается завершения всех допускающих ожидания объектов, прежде чем станет возможен доступ к результатам.

In [11]:
import asyncio, random, time
sushi = 30

async def a_delay(name:str, delay_seconds:float) -> float:
    print(f'{name} будет занят {delay_seconds} с')
    await asyncio.sleep(delay_seconds)
    print(f'{name} закончил есть за {delay_seconds} с')
    return delay_seconds

async def filosofa(name:str, lockA: asyncio.Lock, lockB: asyncio.Lock):
    global sushi
    while sushi > 0: 
      print(f'{name} ждет возможности захватить палочки')
      async with lockA:
        async with lockB:
          print(f'{name} начал есть!')
          sushi -= 1
          await a_delay(name, round(random.uniform(3,6),2))
      print(f'{name} освободил палочки')
      await asyncio.sleep(random.uniform(2,7))

nfs = 5
philoNames = ('1:Кант:','2:Маркс:','3:Платон:','4:Руссо:','5:Сократ:',
              '6:Пифагор:','7:Гегель:','8:Вольтер:','9:Декарт:',
              '10:ЛаоЦзы:','11:Рассел:','12:Лев:')

# создание семафоров ресурсов (палочек, forks) в виде мьютексов
forks = [asyncio.Lock() for _ in range(nfs)]
modelstart=time.perf_counter_ns()
philosofers = [filosofa(philoNames[i], forks[i%nfs], forks[(i+1)%nfs]) for i in range(nfs)]
status = await asyncio.gather( *philosofers)
#
modelfini=time.perf_counter_ns()
print ("== Всё закончилось! Время =", (modelfini-modelstart)/1000000, "msec")

1:Кант: ждет возможности захватить палочки
1:Кант: начал есть!
1:Кант: будет занят 4.6 с
2:Маркс: ждет возможности захватить палочки
3:Платон: ждет возможности захватить палочки
3:Платон: начал есть!
3:Платон: будет занят 3.89 с
4:Руссо: ждет возможности захватить палочки
5:Сократ: ждет возможности захватить палочки
3:Платон: закончил есть за 3.89 с
3:Платон: освободил палочки
1:Кант: закончил есть за 4.6 с
1:Кант: освободил палочки
2:Маркс: начал есть!
2:Маркс: будет занят 5.68 с
5:Сократ: начал есть!
5:Сократ: будет занят 3.09 с
3:Платон: ждет возможности захватить палочки
5:Сократ: закончил есть за 3.09 с
5:Сократ: освободил палочки
4:Руссо: начал есть!
4:Руссо: будет занят 5.84 с
1:Кант: ждет возможности захватить палочки
2:Маркс: закончил есть за 5.68 с
2:Маркс: освободил палочки
1:Кант: начал есть!
1:Кант: будет занят 3.59 с
5:Сократ: ждет возможности захватить палочки
4:Руссо: закончил есть за 5.84 с
4:Руссо: освободил палочки
3:Платон: начал есть!
3:Платон: будет занят 4.86 с
1

Дополните код модели регистрацией _длительности_ интервалов еды каждого философа. 
После завершения выполнения модели выведите в консоль _суммарную длительность_ еды для каждого философа.

Понаблюдайте за результатами при разных количествах философов (от 3 до 11). Выведите в консоль результаты для каждого варианта.

In [2]:
import asyncio, random, time
import nest_asyncio
nest_asyncio.apply()

class EatingStats:
    def __init__(self, name):
        self.name = name
        self.total_eating_time = 0
        self.meals_count = 0
    
    def add_eating(self, duration):
        self.total_eating_time += duration
        self.meals_count += 1

async def a_delay_with_stats(name:str, delay_seconds:float, stats:EatingStats) -> float:
    print(f'{name} будет занят {delay_seconds} с')
    start = time.perf_counter()
    await asyncio.sleep(delay_seconds)
    duration = time.perf_counter() - start
    stats.add_eating(duration)
    print(f'{name} закончил есть за {duration:.2f} с')
    return duration

async def filosofa_with_stats(name:str, lockA: asyncio.Lock, lockB: asyncio.Lock, stats:EatingStats, sushi_dict:dict):
    while sushi_dict['count'] > 0:
        print(f'{name} ждет возможности захватить палочки')
        async with lockA:
            async with lockB:
                if sushi_dict['count'] > 0:
                    print(f'{name} начал есть!')
                    sushi_dict['count'] -= 1
                    eat_time = round(random.uniform(3,6), 2)
                    await a_delay_with_stats(name, eat_time, stats)
        print(f'{name} освободил палочки')
        await asyncio.sleep(random.uniform(2,7))

philoNames = ('1:Кант:','2:Маркс:','3:Платон:','4:Руссо:','5:Сократ:',
              '6:Пифагор:','7:Гегель:','8:Вольтер:','9:Декарт:',
              '10:ЛаоЦзы:','11:Рассел:','12:Лев:')

async def run_for_n_philosophers(num_philosophers, sushi_count=30):
    print(f"\n{'='*60}")
    print(f"Запуск с {num_philosophers} философами, суши: {sushi_count}")
    print(f"{'='*60}\n")
    
    forks = [asyncio.Lock() for _ in range(num_philosophers)]
    stats_list = [EatingStats(philoNames[i]) for i in range(num_philosophers)]
    sushi_dict = {'count': sushi_count}
    
    philosophers = []
    for i in range(num_philosophers):
        philosophers.append(filosofa_with_stats(philoNames[i], forks[i], forks[(i+1)%num_philosophers], stats_list[i], sushi_dict))
    
    modelstart = time.perf_counter_ns()
    await asyncio.gather(*philosophers)
    modelfini = time.perf_counter_ns()
    
    print(f"\n--- Результаты для {num_philosophers} философов ---")
    for i, stats in enumerate(stats_list):
        print(f'{philoNames[i]} поел {stats.meals_count} раз, суммарное время еды: {stats.total_eating_time:.2f} с')
    
    print(f"\n== Всё закончилось! Время = {(modelfini-modelstart)/1000000:.2f} msec")

for n in range(3, 12):
    await run_for_n_philosophers(n, sushi_count=5)


Запуск с 3 философами, суши: 5

1:Кант: ждет возможности захватить палочки
1:Кант: начал есть!
1:Кант: будет занят 3.89 с
2:Маркс: ждет возможности захватить палочки
3:Платон: ждет возможности захватить палочки
1:Кант: закончил есть за 3.90 с
1:Кант: освободил палочки
3:Платон: начал есть!
3:Платон: будет занят 3.59 с
3:Платон: закончил есть за 3.59 с
3:Платон: освободил палочки
2:Маркс: начал есть!
2:Маркс: будет занят 3.98 с
1:Кант: ждет возможности захватить палочки
2:Маркс: закончил есть за 3.98 с
2:Маркс: освободил палочки
1:Кант: начал есть!
1:Кант: будет занят 5.1 с
3:Платон: ждет возможности захватить палочки
2:Маркс: ждет возможности захватить палочки
1:Кант: закончил есть за 5.10 с
1:Кант: освободил палочки
3:Платон: начал есть!
3:Платон: будет занят 4.82 с
3:Платон: закончил есть за 4.83 с
3:Платон: освободил палочки
2:Маркс: освободил палочки

--- Результаты для 3 философов ---
1:Кант: поел 2 раз, суммарное время еды: 9.00 с
2:Маркс: поел 1 раз, суммарное время еды: 3.98 с